# Minimum Spanning Tree - Kruskal

Given a connected, undirected, weighted graph, a **spanning tree** is a subset of the
edges that touches every vertex and contains no cycle: exactly `V - 1` edges. A
**minimum** spanning tree is the cheapest such subset.

Kruskal's answer is almost embarrassingly short: sort the edges by weight and take each
one unless it closes a cycle. Day 14's Union-Find is what makes "does this close a
cycle?" an O(alpha(V)) question.

## Union-Find, from day 14

In [1]:
class UnionFind:
    """Disjoint set union with path compression and union by rank."""

    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.count = n                     # number of components

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:      # path compression
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False                   # already together -> this edge is a cycle
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        self.count -= 1
        return True

## Kruskal

`union` returning `False` *is* the cycle test - no traversal, no visited set.

In [2]:
def kruskal(n, edges):
    """edges: list of (weight, u, v). Returns (chosen, total, trace)."""
    uf = UnionFind(n)
    chosen, total, trace = [], 0, []

    for w, u, v in sorted(edges):          # cheapest first
        if uf.union(u, v):                 # different components -> safe
            chosen.append((w, u, v))
            total += w
            trace.append(('take', w, u, v, uf.count))
            if len(chosen) == n - 1:       # a spanning tree has exactly n-1 edges
                break
        else:
            trace.append(('skip', w, u, v, uf.count))

    return chosen, total, trace


def is_connected(n, edges):
    uf = UnionFind(n)
    for w, u, v in edges:
        uf.union(u, v)
    return uf.count == 1

## The classic 7-vertex graph

Watch the `components left` column: every `take` drops it by one, every `skip` leaves it
alone. That column is the proof that the algorithm terminates after `V - 1` takes.

In [3]:
NAMES = 'ABCDEFG'
GRAPH = [(7, 0, 1), (5, 0, 3), (8, 1, 2), (9, 1, 3), (7, 1, 4),
         (5, 2, 4), (15, 3, 4), (6, 3, 5), (8, 4, 5), (9, 4, 6), (11, 5, 6)]
n = len(NAMES)
show = lambda es: ' '.join('%s%s(%d)' % (NAMES[u], NAMES[v], w) for w, u, v in es)

chosen, total, trace = kruskal(n, GRAPH)
print('sorted edges:', show(sorted(GRAPH)))
print()
print('%-6s %-6s %-7s %s' % ('action', 'edge', 'weight', 'components left'))
for what, w, u, v, comp in trace:
    print('%-6s %-6s %-7d %d' % (what, NAMES[u] + '-' + NAMES[v], w, comp))
print()
print('MST  :', show(chosen))
print('total:', total)

sorted edges: AD(5) CE(5) DF(6) AB(7) BE(7) BC(8) EF(8) BD(9) EG(9) FG(11) DE(15)

action edge   weight  components left
take   A-D    5       6
take   C-E    5       5
take   D-F    6       4
take   A-B    7       3
take   B-E    7       2
skip   B-C    8       2
skip   E-F    8       2
skip   B-D    9       2
take   E-G    9       1

MST  : AD(5) CE(5) DF(6) AB(7) BE(7) EG(9)
total: 39


## Is it really the minimum?

The greedy choice feels too easy, so check it against every possible spanning tree.
This only works because the graph is tiny - the point is the proof, not the method.

In [4]:
from itertools import combinations

best, best_set = None, None
for combo in combinations(GRAPH, n - 1):
    if is_connected(n, combo):
        w = sum(e[0] for e in combo)
        if best is None or w < best:
            best, best_set = w, combo

print('cheapest spanning subset found by brute force:', best)
print('one such tree:', show(sorted(best_set)))
print('Kruskal agrees:', best == total)

cheapest spanning subset found by brute force: 39
one such tree: AD(5) CE(5) DF(6) AB(7) BE(7) EG(9)
Kruskal agrees: True


## Disconnected input gives a spanning *forest*

Nothing in Kruskal assumes connectivity. If the graph falls apart, the loop simply runs
out of usable edges and returns fewer than `V - 1` of them - one tree per component.

In [5]:
broken = [e for e in GRAPH if 6 not in (e[1], e[2])]   # cut G loose
f_edges, f_total, _ = kruskal(n, broken)
print('edges kept:', len(f_edges), '(a spanning tree would need', n - 1, ')')
print('total     :', f_total)
print('G is alone in its own component')

edges kept: 5 (a spanning tree would need 6 )
total     : 30
G is alone in its own component


## LeetCode 1584 - Min Cost to Connect All Points

Every pair of points is an edge whose weight is the Manhattan distance, so the graph is
complete: `n(n-1)/2` edges. Sorting them costs `O(n^2 log n)`, which is fine up to the
problem's `n <= 1000`. For a dense graph like this one, Prim with a heap is the better
asymptotic choice.

In [6]:
def min_cost_connect_points(points):
    n = len(points)
    edges = []
    for i, j in combinations(range(n), 2):
        (x1, y1), (x2, y2) = points[i], points[j]
        edges.append((abs(x1 - x2) + abs(y1 - y2), i, j))
    return kruskal(n, edges)[1]


cases = [([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]], 20),
         ([[3, 12], [-2, 5], [-4, 1]], 18),
         ([[0, 0]], 0)]
for pts, want in cases:
    got = min_cost_connect_points(pts)
    print('%-42s -> %-3d expected %d' % (pts, got, want))

[[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]  -> 20  expected 20
[[3, 12], [-2, 5], [-4, 1]]                -> 18  expected 18
[[0, 0]]                                   -> 0   expected 0


## Tests

In [7]:
assert total == 39 == best
assert len(chosen) == n - 1 and is_connected(n, chosen)
assert len(f_edges) == n - 2
for pts, want in cases:
    assert min_cost_connect_points(pts) == want
print('all assertions passed')

all assertions passed
